In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "455_build_gold_vw_dim_county.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.dim_county"
TARGET_VIEW = f"{catalog}.gold.vw_dim_county"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_county
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          county_key               AS `County Key`
        , county                   AS `County`
        , county_location          AS `County Location`
        , is_official_texas_county AS `Is Official Texas County`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_county successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_county")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise